# Notebook 02 - Data Cleaning

**Purpose:**  
This notebook standardises all six datasets so they share a common structure.  
Every dataset exits this notebook with:
- `county` - plain English county name e.g. `Turkana`
- `pcode` - OCHA 5-character P-code e.g. `KE023`
- `year_month` - period string e.g. `2006-01`

**Column standardisation map:**
| Dataset | County source | PCODE source | Filter |
|---------|--------------|--------------|--------|
| `rainfall` | `PCODE[:5]` mapped via lookup | `PCODE[:5]` | `adm_level == 2` |
| `food_price` | `admin2` | reverse PCODE lookup | actual + retail + staples |
| `conflict_event` | `ADMIN1` | reverse PCODE lookup | political violence only |
| `food_security` | `admin1_name` | `admin1_code` | `admin_level == 1` + `ipc_type == current` |
| `population` | `admin1_name` | `admin1_code` | `admin_level == 1` |
| `poverty_rate` | `admin1_name` | `admin1_code` | `admin_level == 1` |


## 0.0 Setup - Libraries and Configuration

In [126]:
# Standard library imports
import os
import warnings

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Project colour palette
COLOURS = {
    'rainfall':     '#2166ac',   # Blue  - water/climate
    'food_price':   '#d6604d',   # Red   - price stress
    'conflict':     '#1a1a1a',   # Near-black - severity
    'food_security':'#f4a582',   # Salmon - hunger
    'poverty':      '#762a83',   # Purple - structural vulnerability
    'highlight':    '#fdae61',   # Amber  - callout
    'neutral':      '#878787',   # Grey   - background elements
    'high_vuln':    '#b2182b',   # Dark red - high vulnerability counties
}

print("Libraries loaded successfully.")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

Libraries loaded successfully.
pandas  1.4.4
numpy   1.24.4


## 1. Paths and Reference Lookups

In [169]:
# Folder paths
RAW     = os.path.join('..', 'data', 'raw')
CLEAN   = os.path.join('..', 'data', 'processed')
fig_dir = os.path.join('..', 'outputs', 'figures')
tbl_dir = os.path.join('..', 'outputs', 'tables')

os.makedirs(RAW,   exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(tbl_dir, exist_ok=True)

In [128]:
# PCODE to county name lookup
# Used to translate rainfall PCODEs into county names
# and to reverse-map county names back to PCODEs for food price and conflict
PCODE_COUNTY = {
    'KE001': 'Mombasa',         'KE002': 'Kwale',           'KE003': 'Kilifi',
    'KE004': 'Tana River',      'KE005': 'Lamu',            'KE006': 'Taita Taveta',
    'KE007': 'Garissa',         'KE008': 'Wajir',           'KE009': 'Mandera',
    'KE010': 'Marsabit',        'KE011': 'Isiolo',          'KE012': 'Meru',
    'KE013': 'Tharaka-Nithi',   'KE014': 'Embu',            'KE015': 'Kitui',
    'KE016': 'Machakos',        'KE017': 'Makueni',         'KE018': 'Nyandarua',
    'KE019': 'Nyeri',           'KE020': 'Kirinyaga',       'KE021': "Murang'a",
    'KE022': 'Kiambu',          'KE023': 'Turkana',         'KE024': 'West Pokot',
    'KE025': 'Samburu',         'KE026': 'Trans Nzoia',     'KE027': 'Uasin Gishu',
    'KE028': 'Elgeyo-Marakwet', 'KE029': 'Nandi',           'KE030': 'Baringo',
    'KE031': 'Laikipia',        'KE032': 'Nakuru',          'KE033': 'Narok',
    'KE034': 'Kajiado',         'KE035': 'Kericho',         'KE036': 'Bomet',
    'KE037': 'Kakamega',        'KE038': 'Vihiga',          'KE039': 'Bungoma',
    'KE040': 'Busia',           'KE041': 'Siaya',           'KE042': 'Kisumu',
    'KE043': 'Homa Bay',        'KE044': 'Migori',          'KE045': 'Kisii',
    'KE046': 'Nyamira',         'KE047': 'Nairobi'
}

# Reverse lookup: county name -> PCODE
COUNTY_PCODE = {v: k for k, v in PCODE_COUNTY.items()}

# Analysis window
WINDOW_START = '2006-01-01'
WINDOW_END   = '2026-12-31'

# High vulnerability counties (MPI >= 0.30) - from Notebook 01
HIGH_VULN = ['Mandera', 'Marsabit', 'Samburu', 'Tana River', 'Turkana', 'Wajir', 'West Pokot']

print(f"PCODE lookup: {len(PCODE_COUNTY)} counties")
print(f"Analysis window: {WINDOW_START} to {WINDOW_END}")

PCODE lookup: 47 counties
Analysis window: 2006-01-01 to 2026-12-31


## 2. Load Raw Datasets

In [129]:
# Load HDX HAPI supporting datasets
DATE_COLS = ['reference_period_start', 'reference_period_end']

def load_hapi(name):
    return pd.read_csv(
        os.path.join(RAW, f'hdx_hapi_{name}_ken.csv'),
        parse_dates=DATE_COLS
    )

In [130]:
# Load all datasets
DATASETS = {
    'conflict_event': load_hapi('conflict_event'),
    'food_security':  load_hapi('food_security'),
    'population':     load_hapi('population'),
    'poverty_rate':   load_hapi('poverty_rate'),
    'rainfall':       pd.read_csv(os.path.join(RAW, 'ken-rainfall-subnat-full.csv'),  parse_dates=['date']),
    'food_price':     pd.read_csv(os.path.join(RAW, 'wfp_food_prices_ken.csv'),       parse_dates=['date']),
}

for name, df in DATASETS.items():
    print(f"  {name:<20} {len(df):>8,} rows  {len(df.columns)} columns")

  conflict_event         16,150 rows  13 columns
  food_security           4,830 rows  16 columns
  population              3,168 rows  17 columns
  poverty_rate               73 rows  14 columns
  rainfall              132,273 rows  15 columns
  food_price             26,648 rows  16 columns


## 3. Clean Rainfall

**Source:** `ken-rainfall-subnat-full.csv`  
**County column:** `PCODE` (first 5 chars) -> mapped to county name via `PCODE_COUNTY`  
**Filter:** `adm_level == 2` (sub-county rows cover all 47 counties)  
**Aggregate:** 3 dekads per month -> monthly mean `rfq` per county

In [131]:
rain_raw = DATASETS['rainfall'].copy()

# Step 1: filter to sub-county level and analysis window
rain = rain_raw[
    (rain_raw['adm_level'] == 2) &
    (rain_raw['date'] >= WINDOW_START) &
    (rain_raw['date'] <= WINDOW_END)
].copy()

print(f"After filter: {len(rain):,} rows")

# Step 2: derive county and pcode from PCODE column
# Admin2 PCODE format: KE023045 - parent county = first 5 chars = KE023 = Turkana
rain['pcode']  = rain['PCODE'].str[:5]
rain['county'] = rain['pcode'].map(PCODE_COUNTY)

unmapped = rain['county'].isna().sum()
print(f"Unmapped PCODEs: {unmapped}")

# Step 3: create year_month period string
rain['year_month'] = rain['date'].dt.to_period('M').astype(str)

# Step 4: aggregate 3 dekads to monthly mean per county
# rfq_mean = average rainfall anomaly across the 3 dekads
# rfq_min  = worst dekad - captures peak drought severity
rain_clean = (
    rain
    .groupby(['pcode', 'county', 'year_month'])
    .agg(
        rfq_mean=('rfq', 'mean'),
        rfq_min=('rfq',  'min'),
    )
    .reset_index()
    .sort_values(['county', 'year_month'])
    .reset_index(drop=True)
)

print(f"Clean rainfall: {len(rain_clean):,} rows | {rain_clean['county'].nunique()} counties")
print(f"Date range: {rain_clean['year_month'].min()} to {rain_clean['year_month'].max()}")
rain_clean.head()

After filter: 53,509 rows
Unmapped PCODEs: 0
Clean rainfall: 11,515 rows | 47 counties
Date range: 2006-01 to 2026-05


,pcode,county,year_month,rfq_mean,rfq_min
0,KE030,Baringo,2006-01,72.33,56.47
1,KE030,Baringo,2006-02,82.81,39.29
2,KE030,Baringo,2006-03,122.29,50.58
3,KE030,Baringo,2006-04,104.61,61.62
4,KE030,Baringo,2006-05,81.69,49.65


In [132]:
# Validate: confirm 3 dekads per county-month collapsed correctly
raw_dekads = rain.groupby(['county','year_month']).size()
print("Dekads per county-month:")
print(raw_dekads.value_counts().to_dict())
print()

# Validate: rfq_mean should be centred around 100
print(f"rfq_mean - mean: {rain_clean['rfq_mean'].mean():.1f}%  (expected ~100%)")
print(f"rfq_mean - nulls: {rain_clean['rfq_mean'].isna().sum()}")

Dekads per county-month:
{3: 6347, 6: 4148, 9: 732, 12: 244, 1: 26, 2: 17, 4: 1}

rfq_mean - mean: 107.6%  (expected ~100%)
rfq_mean - nulls: 0


## 4. Clean Food Price

**Source:** `wfp_food_prices_ken.csv`  
**County column:** `admin2` (district/county name - already plain English)  
**Filter:** `priceflag == actual`, `pricetype == Retail`, food security staples only  
**Commodities:**
- `Maize (white)` + `Maize (white, dry)` -> merged as `price_maize`
- `Beans` + `Beans (dry)` -> merged as `price_beans`
- `Cowpeas (dry)` -> `price_cowpeas` (ASAL counties, 2023 onwards only)

In [133]:
food_raw = DATASETS['food_price'].copy()

# Commodity map - merge duplicate commodity names into clean labels
# Maize (white) and Maize (white, dry) are the same crop, different unit descriptions
# Beans and Beans (dry) are the same crop, different descriptions across markets
# Sorghum excluded: only 1 actual retail row - 708 rows are wholesale
COMMODITY_MAP = {
    'Maize (white)':      'maize',
    'Maize (white, dry)': 'maize',
    'Beans':              'beans',
    'Beans (dry)':        'beans',
}

# Step 1: filter to actual retail prices, analysis window, target commodities
food = food_raw[
    food_raw['commodity'].isin(COMMODITY_MAP.keys()) &
    (food_raw['priceflag'] == 'actual') &
    (food_raw['pricetype'] == 'Retail') &
    (food_raw['date'] >= WINDOW_START) &
    (food_raw['date'] <= WINDOW_END)
].copy()

print(f"After filter: {len(food):,} rows")

# Step 2: standardise column names
food['commodity_clean'] = food['commodity'].map(COMMODITY_MAP)
food['county']          = food['admin2']   # admin2 is the county name
food['pcode']           = food['county'].map(COUNTY_PCODE)
food['year_month']      = food['date'].dt.to_period('M').astype(str)

# Step 3: drop rows where county could not be mapped to a PCODE
# These are old district names that do not match modern county boundaries
unmapped = food[food['pcode'].isna()][['county']].drop_duplicates()
print(f"\nUnmapped counties (old district names): {len(unmapped)}")
print(unmapped['county'].tolist())

food = food.dropna(subset=['pcode']).copy()
print(f"After dropping unmapped: {len(food):,} rows")

After filter: 2,411 rows

Unmapped counties (old district names): 3
[nan, 'Meru North', 'Moyale']
After dropping unmapped: 2,323 rows


In [134]:
# Step 4: check the row with missing values
food_raw[food_raw['admin2'].isna()].head(2)

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
3440,2015-01-15,NaN,NaN,Hola (Tana River),1854,NaN,NaN,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,35.00,0.38
3484,2015-02-15,NaN,NaN,Hola (Tana River),1854,NaN,NaN,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,35.00,0.38


In [135]:
food_raw['market'][food_raw['admin2'].isna()].unique()

array(['Hola (Tana River)'], dtype=object)

In [136]:
# Step 5: reassign old district names and unnamed markets to correct modern counties
# Moyale    -> Marsabit  (Moyale is in Marsabit county)
# Meru North -> Meru     (Meru North district is now part of Meru county)
# NaN admin2 -> Tana River (all 68 rows are from 'Hola (Tana River)' market — county is in market name)

COUNTY_REMAP = {
    'Moyale':    'Marsabit',
    'Meru North': 'Meru',
}

# Apply remap for known old district names
food['county'] = food['county'].replace(COUNTY_REMAP)

# Assign Tana River to all rows where admin2 is null
# Confirmed: all 68 null admin2 rows are from 'Hola (Tana River)' market
food.loc[food['county'].isna(), 'county'] = 'Tana River'

# Now map pcodes - all counties should resolve
food['pcode'] = food['county'].map(COUNTY_PCODE)

# Confirm no unmapped rows remain
remaining = food[food['pcode'].isna()]
print(f"Remaining unmapped rows: {len(remaining)}")
print(f"Total rows after reassignment: {len(food):,}")
print()
print("County distribution after reassignment:")
print(food['county'].value_counts().to_string())

Remaining unmapped rows: 0
Total rows after reassignment: 2,323

County distribution after reassignment:
Turkana         578
Kitui           360
Marsabit        250
Mandera         224
Garissa         164
Baringo         154
Kilifi          143
Kajiado         140
Tana River       85
Samburu          83
Isiolo           65
Wajir            65
Mombasa           5
Makueni           4
Taita Taveta      2
Nairobi           1


In [137]:
# Step 4: average price per county per commodity per month
# Multiple markets in the same county-month are averaged
food_monthly = (
    food
    .groupby(['pcode', 'county', 'year_month', 'commodity_clean'])['price']
    .mean()
    .reset_index()
)

# Step 5: pivot commodities into columns - one row per county per month
food_clean = food_monthly.pivot_table(
    index=['pcode', 'county', 'year_month'],
    columns='commodity_clean',
    values='price'
).reset_index()

# Clean column names
food_clean.columns = (
    ['pcode', 'county', 'year_month'] +
    [f'price_{c}' for c in food_clean.columns[3:]]
)
food_clean = food_clean.sort_values(['county','year_month']).reset_index(drop=True)

print(f"Clean food price: {len(food_clean):,} rows | {food_clean['county'].nunique()} counties")
print(f"Date range: {food_clean['year_month'].min()} to {food_clean['year_month'].max()}")
print(f"Columns: {food_clean.columns.tolist()}")
print()
print("Null counts per price column:")
for col in [c for c in food_clean.columns if c.startswith('price_')]:
    n = food_clean[col].isna().sum()
    print(f"  {col:<20} {n:>4} nulls  ({n/len(food_clean)*100:.1f}%)")
food_clean.head()

Clean food price: 1,051 rows | 16 counties
Date range: 2006-01 to 2025-12
Columns: ['pcode', 'county', 'year_month', 'price_beans', 'price_maize']

Null counts per price column:
  price_beans           598 nulls  (56.9%)
  price_maize             8 nulls  (0.8%)


,pcode,county,year_month,price_beans,price_maize
0,KE030,Baringo,2015-01,104.00,42.00
1,KE030,Baringo,2015-02,107.00,47.00
2,KE030,Baringo,2015-03,112.00,38.00
3,KE030,Baringo,2015-04,105.00,42.00
4,KE030,Baringo,2015-05,106.00,45.00


In [138]:
# Step 6: handle maize nulls - forward fill within county, max 1 period
# Maize nulls: 0.8% 
# Forward fill within county, max 1 period only
# Rationale: a missed collection month does not mean price changed to zero
food_clean = food_clean.sort_values(['county', 'year_month']).reset_index(drop=True)
food_clean['price_maize'] = (
    food_clean
    .groupby('county')['price_maize']
    .transform(lambda x: x.ffill(limit=1))
)

#Step 7: Handle maize nulls
# Beans nulls: 56.9%

print("Null counts after handling:")
for col in ['price_maize', 'price_beans']:
    n   = food_clean[col].isna().sum()
    pct = n / len(food_clean) * 100
    print(f"  {col:<18} {n:>4} nulls ({pct:.1f}%)")

print()
print("Beans coverage by county:")
coverage = (
    food_clean
    .groupby('county')['price_beans']
    .apply(lambda x: x.notna().mean() * 100)
    .sort_values()
    .apply(lambda x: f"{x:.0f}% coverage")
)
print(coverage.to_string())

pre_2015_nulls = food_clean[
    food_clean['year_month'].str[:4].astype(int) < 2015
]['price_beans'].isna().sum()
total_nulls = food_clean['price_beans'].isna().sum()
share = pre_2015_nulls / total_nulls * 100
print(f"\n {share:.2f}% of missing values are pre-2015")

Null counts after handling:
  price_maize           4 nulls (0.4%)
  price_beans         598 nulls (56.9%)

Beans coverage by county:
county
Marsabit          3% coverage
Mandera           4% coverage
Turkana           4% coverage
Garissa          11% coverage
Isiolo           83% coverage
Tana River       86% coverage
Baringo         100% coverage
Kajiado         100% coverage
Kilifi          100% coverage
Kitui           100% coverage
Makueni         100% coverage
Mombasa         100% coverage
Nairobi         100% coverage
Samburu         100% coverage
Taita Taveta    100% coverage
Wajir           100% coverage

 53.34% of missing values are pre-2015


Beans prices are not imputed due to high structural and regime-based missingness:

- ASAL counties exhibit persistently low observation coverage (<12%), indicating insufficient market depth for reliable price estimation.
- Pre-2015 data reflects incomplete commodity coverage due to phased survey expansion.

Therefore:

- Beans is excluded from imputation procedures
- Analyses involving beans are restricted to adequately observed county-month panels
- Maize is used as the primary stable reference commodity for cascade analysis

## 5. Clean Conflict

**Source:** `hdx_hapi_conflict_event_ken.csv`  
**County column:** `ADMIN1` (already plain English county names)  
**Filter:** political violence only - Battles, Violence against civilians, Explosions  
**Excluded:** Protests, Riots, Strategic developments - different causal mechanism

In [139]:
conf_raw = DATASETS['conflict_event'].copy()

# Dates stored as DD/MM/YYYY strings - must parse with dayfirst=True
conf_raw['date'] = pd.to_datetime(conf_raw['reference_period_start'], dayfirst=True)

print("Event type breakdown (raw):")
print(conf_raw['EVENT_TYPE'].value_counts().to_string())
print()

# Political violence types only
# Rationale: protests and riots are driven by political grievances
# Battles, civilian targeting, and explosions are driven by resource stress
# which is the mechanism we are testing in the cascade
VIOLENCE_TYPES = [
    'Battles',
    'Violence against civilians',
    'Explosions/Remote violence'
]

# Step 1: filter to violence types and analysis window
conf = conf_raw[
    conf_raw['EVENT_TYPE'].isin(VIOLENCE_TYPES) &
    (conf_raw['date'] >= WINDOW_START) &
    (conf_raw['date'] <= WINDOW_END)
].copy()

print(f"After filter: {len(conf):,} rows")
print(f"Event types kept: {conf['EVENT_TYPE'].unique().tolist()}")

Event type breakdown (raw):
Protests                      5300
Riots                         4581
Violence against civilians    2777
Battles                       1783
Strategic developments        1353
Explosions/Remote violence     356

After filter: 4,181 rows
Event types kept: ['Violence against civilians', 'Battles', 'Explosions/Remote violence']


In [140]:
# Step 2: standardise column names
conf['county']     = conf['ADMIN1']
conf['pcode']      = conf['county'].map(COUNTY_PCODE)
conf['year_month'] = conf['date'].dt.to_period('M').astype(str)

# Check unmapped counties
unmapped_conf = conf[conf['pcode'].isna()]['county'].dropna().unique()
print(f"Unmapped conflict counties: {len(unmapped_conf)}")
if len(unmapped_conf) > 0:
    print(unmapped_conf)


Unmapped conflict counties: 2
['Muranga' 'Elgeyo Marakwet']


In [141]:
# Step 3: Fix county name spelling to match PCODE_COUNTY master list
# standardise column names
conf['county'] = conf['ADMIN1']

# fix names FIRST
conflict_name_fix = {
    'Muranga': "Murang'a",
    'Elgeyo Marakwet': 'Elgeyo-Marakwet',
}

conf['county'] = conf['county'].replace(conflict_name_fix)

# THEN map PCODE
conf['pcode'] = conf['county'].map(COUNTY_PCODE)

conf['year_month'] = conf['date'].dt.to_period('M').astype(str)

# NOW check unmapped
unmapped_conf = conf[conf['pcode'].isna()]['county'].dropna().unique()
print(f"Unmapped conflict counties: {len(unmapped_conf)}")
print(unmapped_conf)

Unmapped conflict counties: 0
[]


In [142]:
# Step 4: aggregate to monthly county totals
conf_clean = (
    conf
    .groupby(['pcode', 'county', 'year_month'])
    .agg(
        conflict_events=('EVENTS',     'sum'),
        conflict_fatalities=('FATALITIES', 'sum')
    )
    .reset_index()
    .sort_values(['county', 'year_month'])
    .reset_index(drop=True)
)

print(f"\nClean conflict: {len(conf_clean):,} rows | {conf_clean['county'].nunique()} counties")
print(f"Date range: {conf_clean['year_month'].min()} to {conf_clean['year_month'].max()}")
print(f"Total events: {conf_clean['conflict_events'].sum():,}")
print(f"Total fatalities: {conf_clean['conflict_fatalities'].sum():,}")
conf_clean.head()


Clean conflict: 2,552 rows | 47 counties
Date range: 2006-01 to 2026-11
Total events: 5,103
Total fatalities: 9,171


,pcode,county,year_month,conflict_events,conflict_fatalities
0,KE030,Baringo,2006-08,1,5
1,KE030,Baringo,2007-07,1,0
2,KE030,Baringo,2007-10,1,2
3,KE030,Baringo,2007-11,2,0
4,KE030,Baringo,2007-12,1,9


## 6. Clean Food Security

**Source:** `hdx_hapi_food_security_ken.csv`  
**County column:** `admin1_name` (plain English) | PCODE: `admin1_code`  
**Filter:** `admin_level == 1` AND `admin1_name notna()` AND `ipc_type == current`  
**Note:** `provider_admin1_name` contains groupings like `ASAL Counties` - not counties, ignored

In [ ]:
fs_raw = DATASETS['food_security'].copy()

print("admin_level breakdown (raw):")
print(fs_raw['admin_level'].value_counts().to_dict())
print()
print("ipc_type values:")
print(fs_raw['ipc_type'].value_counts().to_dict())
print()
print("ipc_phase values:")
print(fs_raw['ipc_phase'].value_counts().to_dict())
print()
print("missing counties:")
print(fs_raw['admin1_name'].isna().sum())

admin_level breakdown (raw):
{1: 4648, 0: 182}

ipc_type values:
{'current': 2415, 'first projection': 2415}

ipc_phase values:
{'all': 690, '3+': 690, '1': 690, '2': 690, '3': 690, '4': 690, '5': 690}

missing values values:
588


In [157]:
#Try filling the missing counties
missing_admin = fs_raw.loc[
    fs_raw['admin1_name'].isna(),
    ['admin1_name', 'provider_admin1_name']
]
missing_with_provider = missing_admin[
    missing_admin['provider_admin1_name'].notna()
]
unique_provider_admin = missing_with_provider['provider_admin1_name'].unique()

print(unique_provider_admin)
print(len(unique_provider_admin))

['Others' 'ASAL Counties' 'Urban Analysis' 'Marsabit - moyale'
 'Marsabit - laisamis' 'Marsabit - saku' 'Turkana south'
 'Turkana east-kibish-loima' 'Marsabit - north horr' 'Turkana central'
 'Turkana north' 'KIBRA' 'MUKURU' 'MATHARE' 'KONDELE' 'KANGEMI'
 'KAWANGWARE' 'BANGLADESH' 'KAYOLE' 'DANDORA' 'GITHURAI' 'MWEMBE TAYARI'
 'OBUNGA']
23


In [158]:
# Assign admin1_name from provider_admin1_name where admin1_name is null
# Each group below has been verified against Kenya county boundaries

PROVIDER_COUNTY_MAP = {
    # Marsabit sub-divisions (IPC reported at sub-county level for Marsabit)
    'Marsabit - moyale':        'Marsabit',
    'Marsabit - laisamis':      'Marsabit',
    'Marsabit - saku':          'Marsabit',
    'Marsabit - north horr':    'Marsabit',

    # Turkana sub-divisions
    'Turkana south':            'Turkana',
    'Turkana central':          'Turkana',
    'Turkana north':            'Turkana',
    'Turkana east-kibish-loima':'Turkana',

    # Nairobi informal settlements
    'KIBRA':        'Nairobi',
    'MUKURU':       'Nairobi',
    'MATHARE':      'Nairobi',
    'KONDELE':      'Nairobi',
    'KANGEMI':      'Nairobi',
    'KAWANGWARE':   'Nairobi',
    'BANGLADESH':   'Nairobi',
    'KAYOLE':       'Nairobi',
    'DANDORA':      'Nairobi',
    'GITHURAI':     'Nairobi',

    # Other urban areas
    'MWEMBE TAYARI': 'Mombasa',
    'OBUNGA':        'Kisumu',
}

# Fill admin1_name where null using provider_admin1_name map
mask = fs['admin1_name'].isna() & fs['provider_admin1_name'].isin(PROVIDER_COUNTY_MAP)
fs.loc[mask, 'admin1_name'] = fs.loc[mask, 'provider_admin1_name'].map(PROVIDER_COUNTY_MAP)

# Drop rows that are groupings not assignable to a county
DROP_GROUPS = ['Others', 'ASAL Counties', 'Urban Analysis']
fs = fs[~fs['provider_admin1_name'].isin(DROP_GROUPS) | fs['admin1_name'].notna()].copy()

print(f"After provider name assignment:")
print(f"  Rows remaining:       {len(fs):,}")
print(f"  admin1_name nulls:    {fs['admin1_name'].isna().sum()}")
print()
print("Newly recovered counties:")
print(fs[fs['provider_admin1_name'].isin(PROVIDER_COUNTY_MAP)]['admin1_name'].value_counts().to_string())

After provider name assignment:
  Rows remaining:       2,121
  admin1_name nulls:    0

Newly recovered counties:
Series([], )


In [159]:
# Step 1: filter to county level, current classifications, valid county names
# admin_level == 0 = national aggregate rows - drop
# ipc_type == 'first projection' = forecasts - keep only 'current' for analysis
# drop null rows at level 1 are national aggregates
fs = fs_raw[
    (fs_raw['admin_level'] == 1) &
    (fs_raw['admin1_name'].notna()) &
    (fs_raw['ipc_type'] == 'current')
].copy()

print(f"After filter: {len(fs):,} rows")

# Step 2: standardise column names
# admin1_name is the county, admin1_code is the PCODE
fs['county']     = fs['admin1_name']
fs['pcode']      = fs['admin1_code']
fs['year_month'] = pd.to_datetime(fs['reference_period_start']).dt.to_period('M').astype(str)

# Step 3: keep only analytical columns
fs_clean = (
    fs[['pcode', 'county', 'year_month', 'ipc_phase', 'population_in_phase', 'population_fraction_in_phase']]
    .copy()
    .sort_values(['county', 'year_month', 'ipc_phase'])
    .reset_index(drop=True)
)

print(f"Clean food security: {len(fs_clean):,} rows | {fs_clean['county'].nunique()} counties")
print(f"Date range: {fs_clean['year_month'].min()} to {fs_clean['year_month'].max()}")
print(f"IPC phases: {sorted(fs_clean['ipc_phase'].unique())}")
fs_clean.head(8)

After filter: 2,121 rows
Clean food security: 2,121 rows | 28 counties
Date range: 2019-07 to 2025-07
IPC phases: ['1', '2', '3', '3+', '4', '5', 'all']


,pcode,county,year_month,ipc_phase,population_in_phase,population_fraction_in_phase
0,KE030,Baringo,2019-07,1,246294,0.35
1,KE030,Baringo,2019-07,2,351849,0.50
2,KE030,Baringo,2019-07,3,70370,0.10
3,KE030,Baringo,2019-07,3+,105555,0.15
4,KE030,Baringo,2019-07,4,35185,0.05
5,KE030,Baringo,2019-07,5,0,0.00
6,KE030,Baringo,2019-07,all,703697,1.00
7,KE030,Baringo,2020-02,1,466748,0.70


## 7. Clean Population

**Source:** `hdx_hapi_population_ken.csv`  
**County column:** `admin1_name` | PCODE: `admin1_code`  
**Filter:** `admin_level == 1`  
**Use:** Per capita normalisation scalar - total population per county

In [164]:
pop_raw = DATASETS['population'].copy()

# Filter to county level
pop = pop_raw[pop_raw['admin_level'] == 1].copy()

# Keep only the all-gender all-age row per county
# gender='all' + age_range='all' = total county population
# All other rows are breakdowns that would inflate the sum if included
pop = pop[
    (pop['gender']    == 'all') &
    (pop['age_range'] == 'all')
].copy()

# Standardise column names
pop_clean = (
    pop[['admin1_code', 'admin1_name', 'population']]
    .rename(columns={
        'admin1_name': 'county',
        'admin1_code': 'pcode',
        'population':  'total_population'
    })
    .sort_values('county')
    .reset_index(drop=True)
)

print(f"Clean population: {len(pop_clean)} counties")
print(f"Total Kenya population: {pop_clean['total_population'].sum():,.0f}")
# Expected: ~54 million (2019 census)

Clean population: 47 counties
Total Kenya population: 47,562,772


## 8. Clean Poverty Rate

**Source:** `hdx_hapi_poverty_rate_ken.csv`  
**County column:** `admin1_name` | PCODE: `admin1_code`  
**Filter:** `admin_level == 1`, most recent survey year per county  
**Use:** Static vulnerability tier - joined as county-level label, not time series

In [165]:
pov_raw = DATASETS['poverty_rate'].copy()

# Step 1: filter to county level
pov = pov_raw[pov_raw['admin_level'] == 1].copy()
print(f"Survey years available: {sorted(pov['reference_period_start'].dt.year.unique())}")

# Step 2: keep most recent survey year per county
pov_clean = (
    pov
    .sort_values('reference_period_start', ascending=False)
    .dropna(subset=['admin1_name', 'mpi'])
    .drop_duplicates(subset='admin1_name', keep='first')
    .copy()
)

# Step 3: standardise column names
pov_clean['county'] = pov_clean['admin1_name']
pov_clean['pcode']  = pov_clean['admin1_code']

# Step 4: assign vulnerability tier
def assign_tier(mpi):
    if mpi >= 0.30:   return 'High'    # Chronically deprived - ASAL counties
    elif mpi >= 0.15: return 'Medium'  # Moderate deprivation
    else:             return 'Low'     # Relatively better off

pov_clean['vulnerability_tier'] = pov_clean['mpi'].apply(assign_tier)

# Step 5: keep only needed columns
pov_clean = (
    pov_clean[['pcode', 'county', 'mpi', 'headcount_ratio',
               'intensity_of_deprivation', 'in_severe_poverty', 'vulnerability_tier']]
    .sort_values('mpi', ascending=False)
    .reset_index(drop=True)
)

print(f"\nClean poverty: {len(pov_clean)} counties")
print(f"Tier distribution: {pov_clean['vulnerability_tier'].value_counts().to_dict()}")
print()
print("High vulnerability counties:")
print(pov_clean[pov_clean['vulnerability_tier']=='High'][['county','pcode','mpi']].to_string(index=False))
pov_clean.head()

Survey years available: [2008, 2014, 2022]

Clean poverty: 47 counties
Tier distribution: {'Low': 32, 'Medium': 8, 'High': 7}

High vulnerability counties:
    county pcode  mpi
   Turkana KE023 0.50
   Mandera KE009 0.46
   Samburu KE025 0.42
     Wajir KE008 0.40
Tana River KE004 0.38
West Pokot KE024 0.35
  Marsabit KE010 0.35


,pcode,county,mpi,headcount_ratio,intensity_of_deprivation,in_severe_poverty,vulnerability_tier
0,KE023,Turkana,0.50,79.55,62.45,64.89,High
1,KE009,Mandera,0.46,81.34,56.70,55.90,High
2,KE025,Samburu,0.42,70.16,59.46,53.65,High
3,KE008,Wajir,0.40,73.11,54.11,42.86,High
4,KE004,Tana River,0.38,67.29,56.17,44.42,High


## 9. Verify Standardised Column Names

Every clean dataset must have `county`, `pcode`, and `year_month` (where applicable).  
This cell confirms the standardisation worked before we save anything.

In [166]:
CLEAN_DATASETS = {
    'rainfall':      rain_clean,
    'food_price':    food_clean,
    'conflict':      conf_clean,
    'food_security': fs_clean,
    'population':    pop_clean,
    'poverty_rate':  pov_clean,
}

print(f"{'DATASET':<18} {'ROWS':>8} {'COUNTIES':>10} {'county':>8} {'pcode':>8} {'year_month':>12}")
print("-" * 70)

for name, df in CLEAN_DATASETS.items():
    has_county     = 'county'     in df.columns
    has_pcode      = 'pcode'      in df.columns
    has_year_month = 'year_month' in df.columns
    n_counties     = df['county'].nunique() if has_county else '-'

    print(
        f"{name:<18} {len(df):>8,} {str(n_counties):>10}"
        f" {'YES' if has_county else 'NO':>8}"
        f" {'YES' if has_pcode else 'NO':>8}"
        f" {'YES' if has_year_month else 'N/A (static)':>12}"
    )

DATASET                ROWS   COUNTIES   county    pcode   year_month
----------------------------------------------------------------------
rainfall             11,515         47      YES      YES          YES
food_price            1,051         16      YES      YES          YES
conflict              2,552         47      YES      YES          YES
food_security         2,121         28      YES      YES          YES
population               47         47      YES      YES N/A (static)
poverty_rate             47         47      YES      YES N/A (static)


In [167]:
# County name consistency check
# All datasets should resolve to the same 47 county names
print("County names in each dataset vs PCODE_COUNTY master list")
print(" ")

master_counties = set(PCODE_COUNTY.values())

for name, df in CLEAN_DATASETS.items():
    counties    = set(df['county'].dropna().unique())
    not_in_master = counties - master_counties
    status      = 'CLEAN' if not not_in_master else f'CHECK: {not_in_master}'
    print(f"  {name:<18} {len(counties):>2} counties  {status}")

County names in each dataset vs PCODE_COUNTY master list
 
  rainfall           47 counties  CLEAN
  food_price         16 counties  CLEAN
  conflict           47 counties  CLEAN
  food_security      28 counties  CLEAN
  population         47 counties  CLEAN
  poverty_rate       47 counties  CLEAN


## 10. Save Clean Datasets

In [170]:
# Save each clean dataset to data/processed/
for name, df in CLEAN_DATASETS.items():
    path = os.path.join(CLEAN, f'02_{name}_clean.csv')
    df.to_csv(path, index=False)
    print(f"Saved: data/processed/02_{name}_clean.csv  ({len(df):,} rows)")

Saved: data/processed/02_rainfall_clean.csv  (11,515 rows)
Saved: data/processed/02_food_price_clean.csv  (1,051 rows)
Saved: data/processed/02_conflict_clean.csv  (2,552 rows)
Saved: data/processed/02_food_security_clean.csv  (2,121 rows)
Saved: data/processed/02_population_clean.csv  (47 rows)
Saved: data/processed/02_poverty_rate_clean.csv  (47 rows)


## 11. Build the Analysis Panel

Join all clean time-series datasets into one panel.  
Rainfall is the base (all 47 counties, every month 2006-2026).  
Food price and conflict are left-joined - nulls are data gaps, not errors.  
Poverty and population are county-level labels - joined once, not by month.

In [171]:
# Build panel: rainfall (base) + food price + conflict
panel = rain_clean.merge(food_clean,  on=['pcode','county','year_month'], how='left')
panel = panel.merge(conf_clean, on=['pcode','county','year_month'], how='left')

# Conflict nulls = zero events that month (absence of events, not missing data)
panel['conflict_events']     = panel['conflict_events'].fillna(0).astype(int)
panel['conflict_fatalities'] = panel['conflict_fatalities'].fillna(0).astype(int)

# Add poverty tier and population as county-level labels
panel = panel.merge(
    pov_clean[['pcode','county','mpi','vulnerability_tier']],
    on=['pcode','county'], how='left'
)
panel = panel.merge(
    pop_clean[['pcode','county','total_population']],
    on=['pcode','county'], how='left'
)

panel = panel.sort_values(['county','year_month']).reset_index(drop=True)

print(f"FULL PANEL: {len(panel):,} rows | {panel['county'].nunique()} counties | {len(panel.columns)} columns")
print(f"Columns: {panel.columns.tolist()}")
print()
print("Price column null counts (nulls = no market data for that county):")
for col in [c for c in panel.columns if c.startswith('price_')]:
    n = panel[col].isna().sum()
    print(f"  {col:<20} {n:>5,} nulls ({n/len(panel)*100:.1f}%)")

FULL PANEL: 11,515 rows | 47 counties | 12 columns
Columns: ['pcode', 'county', 'year_month', 'rfq_mean', 'rfq_min', 'price_beans', 'price_maize', 'conflict_events', 'conflict_fatalities', 'mpi', 'vulnerability_tier', 'total_population']

Price column null counts (nulls = no market data for that county):
  price_beans          11,062 nulls (96.1%)
  price_maize          10,468 nulls (90.9%)


In [172]:
# Analysis subset: only counties where food price data exists
# These 18 counties have both rainfall and food price data for the full cascade analysis
analysis_panel = panel.dropna(subset=['price_maize','price_beans'], how='all').copy()

print(f"ANALYSIS PANEL: {len(analysis_panel):,} rows | {analysis_panel['county'].nunique()} counties")
print(f"Counties: {sorted(analysis_panel['county'].unique())}")
print()

# Confirm high vulnerability counties are present
hv_in_analysis = [c for c in HIGH_VULN if c in analysis_panel['county'].unique()]
print(f"High vulnerability counties in analysis panel: {len(hv_in_analysis)}/7")
print(hv_in_analysis)

ANALYSIS PANEL: 1,051 rows | 16 counties
Counties: ['Baringo', 'Garissa', 'Isiolo', 'Kajiado', 'Kilifi', 'Kitui', 'Makueni', 'Mandera', 'Marsabit', 'Mombasa', 'Nairobi', 'Samburu', 'Taita Taveta', 'Tana River', 'Turkana', 'Wajir']

High vulnerability counties in analysis panel: 6/7
['Mandera', 'Marsabit', 'Samburu', 'Tana River', 'Turkana', 'Wajir']


In [173]:
# Save both panels
panel.to_csv(os.path.join(CLEAN, '02_panel_full.csv'), index=False)
analysis_panel.to_csv(os.path.join(CLEAN, '02_panel_analysis.csv'), index=False)

print("Saved: data/processed/02_panel_full.csv     (47 counties, every month)")
print("Saved: data/processed/02_panel_analysis.csv (18 counties with price data)")

Saved: data/processed/02_panel_full.csv     (47 counties, every month)
Saved: data/processed/02_panel_analysis.csv (18 counties with price data)
